# Solution of Bùi Hồng Phúc
Trong bài toán này là bài toán rủi ro tính dụng nên sẽ tập trung và xử lý dữ liệu local thinking về các thông tin của bài toán hơn là chú tâm vào việc xứ lý và tối ưu xử dụng model vậy ta sẽ suy nghĩa và trích lọc các feature để tăng độ chính xác của bài toàn sử dụng thư việc polar. Và sau đây là danh sách các bài bước thực hiện :

In this notebook you will see how to:
* Load the data
* Join tables with Polars - a DataFrame library implemented in Rust language, designed to be blazingy fast and memory efficient.  
* Create features
* Train a LightGBM model
* Create a submission tabl

## Load data

In [1]:
import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score 

dataPath = "/kaggle/input/competitions/home-credit-credit-risk-model-stability/"

In [2]:
# ============================================================
# CELL NÀY PHẢI CHẠY TRƯỚC TẤT CẢ — định nghĩa lại hàm sạch
# ============================================================
import polars as pl
import gc
import os
import glob

def set_table_dtypes(df: pl.LazyFrame) -> pl.LazyFrame:
    schema = df.collect_schema()
    exprs = []

    for col_name in schema.names():
        if col_name == "case_id":
            exprs.append(pl.col(col_name))

        elif col_name.startswith("num_group"):
            exprs.append(pl.col(col_name))

        elif col_name.endswith(("P", "A")):
            exprs.append(
                pl.col(col_name).cast(pl.Float32, strict=False)
            )

        elif col_name.endswith("D"):
            exprs.append(pl.col(col_name))

        elif col_name.endswith(("M", "T")):
            exprs.append(
                pl.col(col_name).cast(pl.Utf8, strict=False)
            )

        elif col_name.endswith("L"):
            # L có thể là numeric hoặc categorical
            # Giữ nguyên để tránh lỗi string > numeric
            exprs.append(pl.col(col_name))

        else:
            exprs.append(pl.col(col_name))

    return df.with_columns(exprs)

def downcast_df(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col_name, dtype in zip(df.columns, df.dtypes):
        if dtype == pl.Float64:
            exprs.append(pl.col(col_name).cast(pl.Float32))
        elif dtype == pl.Int64:
            exprs.append(pl.col(col_name).cast(pl.Int32))
        elif dtype == pl.Int32:
            exprs.append(pl.col(col_name).cast(pl.Int16))
        else:
            exprs.append(pl.col(col_name))
    return df.with_columns(exprs)

def convert_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object" or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype("category")
    return df

def safe_scan(pattern: str) -> pl.LazyFrame:
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"Không tìm thấy: {pattern}\n"
            f"Files có trong thư mục: {sorted(os.listdir(os.path.dirname(pattern)))}"
        )
    print(f"  ✓ {len(files)} file: {[os.path.basename(f) for f in files]}")
    return pl.scan_parquet(files)

def check_memory(label=""):
    import psutil
    ram = psutil.virtual_memory()
    print(f"[{label}] RAM: {ram.used/1e9:.1f}GB / {ram.total/1e9:.1f}GB ({ram.percent}%)")

# Xác nhận hàm đã được load đúng
import inspect
src = inspect.getsource(set_table_dtypes)
assert "df.columns" not in src, "❌ Vẫn còn code cũ!"
print("✅ Tất cả hàm đã được định nghĩa đúng, không còn warning")

✅ Tất cả hàm đã được định nghĩa đúng, không còn warning


### Đọc bản dữ liệu

Trong phần này load toàn bộ dữ liệu lên trên máy

In [3]:
check_memory("START")

train_basetable = pl.scan_parquet(dataPath + "parquet_files/train/train_base.parquet")
train_static = pl.scan_parquet(dataPath + "parquet_files/train/train_static_0_*.parquet").pipe(set_table_dtypes)

train_static_cb = pl.scan_parquet(dataPath + "parquet_files/train/train_static_cb_0.parquet").pipe(set_table_dtypes)
train_person_1 = pl.scan_parquet(dataPath + "parquet_files/train/train_person_1.parquet").pipe(set_table_dtypes) 
train_credit_bureau_a_2 = pl.scan_parquet(dataPath + "parquet_files/train/train_credit_bureau_a_2_*.parquet").pipe(set_table_dtypes) 
train_credit_bureau_b_1 = pl.scan_parquet(dataPath + "parquet_files/train/train_credit_bureau_b_1.parquet").pipe(set_table_dtypes) 
train_credit_bureau_b_2 = pl.scan_parquet(dataPath + "parquet_files/train/train_credit_bureau_b_2.parquet").pipe(set_table_dtypes) 
train_credit_bureau_a_1 = pl.scan_parquet(dataPath + "parquet_files/train/train_credit_bureau_a_1_*.parquet").pipe(set_table_dtypes)
train_tax_registry_a_1  = pl.scan_parquet(dataPath + "parquet_files/train/train_tax_registry_a_1.parquet").pipe(set_table_dtypes)
train_applprev_1 = pl.scan_parquet(dataPath + "parquet_files/train/train_applprev_1_*.parquet").pipe(set_table_dtypes)

[START] RAM: 1.1GB / 33.7GB (4.6%)


In [4]:
test_basetable = pl.scan_parquet(dataPath + "parquet_files/test/test_base.parquet")
test_static = pl.scan_parquet(dataPath + "parquet_files/test/test_static_0_*.parquet").pipe(set_table_dtypes)
test_static_cb = pl.scan_parquet(dataPath + "parquet_files/test/test_static_cb_0.parquet").pipe(set_table_dtypes)
test_person_1 = pl.scan_parquet(dataPath + "parquet_files/test/test_person_1.parquet").pipe(set_table_dtypes) 
test_credit_bureau_a_2 = pl.scan_parquet(dataPath + "parquet_files/test/test_credit_bureau_a_2_*.parquet").pipe(set_table_dtypes)
test_credit_bureau_b_2 = pl.scan_parquet(dataPath + "parquet_files/test/test_credit_bureau_b_2.parquet").pipe(set_table_dtypes)
test_credit_bureau_b_1 = pl.scan_parquet(dataPath + "parquet_files/test/test_credit_bureau_b_1.parquet").pipe(set_table_dtypes)
test_credit_bureau_a_1 = pl.scan_parquet(dataPath + "parquet_files/test/test_credit_bureau_a_1_*.parquet").pipe(set_table_dtypes)
test_tax_registry_a_1 = pl.scan_parquet(dataPath + "parquet_files/test/test_tax_registry_a_1.parquet").pipe(set_table_dtypes) 
test_applprev_1 = pl.scan_parquet(dataPath + "parquet_files/test/test_applprev_1_*.parquet").pipe(set_table_dtypes)

## Feature engineering
Trong phần này mình sẽ thực hiện trích xuất ra các đặc trưng dựa vào dữ liệu mà ta có thể thấy bằng việc tổng hợp và xử lý lại dữ liệu:
1. Ta sẽ xem thu nhập tài chính lớn nhất mà một khách hàng làm nghề tự do có được. Vì việc làm tự do với mức thu nhập không cố định có thể tạo ra các rủi ro về mặt tài chính từ đó cần được quan tâm tới.
2. Ta sẽ xét về mặt nhà ở của người chủ chính đứng ra làm hợp đồng cho thuê. Đánh giá sự ổn định của một khách hàng nếu họ không có nhà hay nhà thuê rủi ro họ bốc hơi là có khả năng.
3. Xem xét xem mức nợ quá hạn cao nhất mà người đó chi trả một người từng nợ quá hạn thấp rủi ro tài chính sẽ thấp hơn. Việc khách hàng trễ hạn chi trả 1 tháng > 31 ngày không thể nào là quên mà là chứng tỏ dòng tiền của họ bị đứt gãy và họ hiện tại không có khả năng chi trả rủi ro dồn nợ là rất cao.
4. cũng cần xem thêm tỷ lệ nợ trên thu nhập
5. Xu hướng xấu đi của hành vi thanh toán xem xem số ngày quá hạn của tháng này thấp hơn tháng trước không ? Xem xem mà max DPD gần đây của khách hàng hay là tỷ lệ trung bình phần trăm các khoản trả góp được thanh toán từ 1 ngày trở lên sau ngày đáo hạn.
6. Xem xét về mức độ sử dụng tính dụng

#### Setup bộ công cụ hỗ trợ tìm kiếm

In [5]:
import polars as pl

def has_column(df: pl.DataFrame, column_name: str) -> bool:
    """
    Kiểm tra một tên cột có tồn tại trong DataFrame hay không.
    Trả về True nếu có, False nếu không.
    """
    # Cách nhanh nhất trong Polars là kiểm tra trong danh sách df.columns hoặc df.schema
    if column_name in df.columns:
        print(f"✅ Cột '{column_name}' CÓ tồn tại trong bảng.")
        return True
    else:
        print(f"❌ Cột '{column_name}' KHÔNG tồn tại trong bảng.")
        return False

In [6]:
import glob
import os
import polars as pl


def find_column_in_folder(target_column: str, folder_path: str) -> list:
    """Hàm quét toàn bộ file .csv trong thư mục để tìm tên cột chỉ định.

    Chỉ đọc schema đầu file nên tốc độ cực nhanh và siêu tiết kiệm RAM.
    """
    # Lấy đường dẫn tuyệt đối của tất cả các file .csv
    search_pattern = os.path.join(folder_path, "*.csv")
    csv_files = glob.glob(search_pattern)

    matched_files = []

    print(f"🕵️ Đang tiến hành quét {len(csv_files)} file .csv...")

    for file_path in csv_files:
        try:
            # scan_csv chỉ đọc cấu trúc cột (schema) chứ không load dữ liệu dòng
            schema = pl.scan_csv(file_path).schema

            if target_column in schema:
                # Tách lấy tên file để hiển thị cho gọn thay vì lấy đường dẫn dài
                file_name = os.path.basename(file_path)
                matched_files.append(file_name)

        except Exception as e:
            # Bỏ qua và báo lỗi nếu gặp file rỗng hoặc bị hỏng cấu trúc
            print(f"⚠️ Lỗi khi đọc cấu trúc file {os.path.basename(file_path)}: {e}")

    # In kết quả tổng hợp ra màn hình
    if matched_files:
        print(
            f"\n✅ Tìm thấy cột '{target_column}' xuất hiện trong {len(matched_files)} file sau:"
        )
        for f in sorted(matched_files):
            print(f"   🔹 {f}")
    else:
        print(
            f"\n❌ Không tìm thấy cột '{target_column}' trong bất kỳ file .csv nào tại thư mục này."
        )

    return matched_files


# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN VÀ CHẠY THỬ NGHIỆM
# ---------------------------------------------------------
KAGGLE_TRAIN_DIR = "/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train"
# Ví dụ 2: Tìm cột tổng nợ quá hạn
# found_files = find_column_in_folder("totaldebtoverduevalue_178A", KAGGLE_TRAIN_DIR)

### Thực hiện thiết kế data

#### Thực hiện các rủ ro tài chính cơ bản từ mẫu

In [7]:
# We need to use aggregation functions in tables with depth > 1, so tables that contain num_group1 column or 
# also num_group2 column.
train_person_1_feats_1 = train_person_1.group_by("case_id").agg(
    pl.col("mainoccupationinc_384A").max().alias("mainoccupationinc_384A_max"),
    (pl.col("incometype_1044T") == "SELFEMPLOYED").max().alias("mainoccupationinc_384A_any_selfemployed")
)

# Here num_group1=0 has special meaning, it is the person who applied for the loan.
train_person_1_feats_2 = train_person_1.select(["case_id", "num_group1", "housetype_905L"]).filter(
    pl.col("num_group1") == 0
).drop("num_group1").rename({"housetype_905L": "person_housetype"})

# Here we have num_goup1 and num_group2, so we need to aggregate again.
train_credit_bureau_b_2_feats = train_credit_bureau_b_2.group_by("case_id").agg(
    pl.col("pmts_pmtsoverdue_635A").max().alias("pmts_pmtsoverdue_635A_max"),
    (pl.col("pmts_dpdvalue_108P") > 31).max().alias("pmts_dpdvalue_108P_over31")
)



In [8]:
test_person_1_feats_1 = test_person_1.group_by("case_id").agg(
    pl.col("mainoccupationinc_384A").max().alias("mainoccupationinc_384A_max"),
    (pl.col("incometype_1044T") == "SELFEMPLOYED").max().alias("mainoccupationinc_384A_any_selfemployed")
)

test_person_1_feats_2 = test_person_1.select(["case_id", "num_group1", "housetype_905L"]).filter(
    pl.col("num_group1") == 0
).drop("num_group1").rename({"housetype_905L": "person_housetype"})

test_credit_bureau_b_2_feats = test_credit_bureau_b_2.group_by("case_id").agg(
    pl.col("pmts_pmtsoverdue_635A").max().alias("pmts_pmtsoverdue_635A_max"),
    (pl.col("pmts_dpdvalue_108P") > 31).max().alias("pmts_dpdvalue_108P_over31")
)

#### Debt Burden & Affordability (Gánh nặng nợ)


trong phần này sẽ xử lý gánh nặng nợ  ta sẽ xử lý để lấy ra thu nhập của người vay chính và nợ quá hạn lớn nhât của người đó sau đó xử lý:

1. Tính tỷ lệ dựa trên tổng nợ/ thu nhập chính của khách hàng để đánh giá được sự chệch lệch giữa thu nhập và nợ có đồng đều hay không ?
2. Xem xét về khả năng trả nợ hàng tháng.
3. Tính tỷ lệ tổng nợ đang quá hạn và tổng dư nợ để biết được mức độ nghiêm trọng của sự cố đỗ vỡ tài chính ví dụ:
A: tổng dư nợ là 2 tỷ đồng nợ quá hạn là 10 thì rủi ro sẽ thấp hơn người tổng nợ 2 tỷ mà nợ quá hạn là 200tr

In [9]:
# Debt-to-Income Ratio (DTI) - Chỉ số vàng trong tín dụng
# 1.1  lấy thu nhập của NGƯỜI VAY CHÍNH (num_group1 = 0)
person_flat = train_person_1.filter(pl.col("num_group1") == 0).select([
    "case_id", 
    "mainoccupationinc_384A"
])

# 1.2  nhóm lấy Tổng (hoặc Max) nợ quá hạn
# (Giả định cột này nằm ở bảng credit_bureau_a_1 như code bạn viết)
cb_a_1_flat = train_credit_bureau_a_1.group_by("case_id").agg([
    pl.col("totaldebtoverduevalue_178A").max().alias("totaldebtoverduevalue_178A_max"),
    pl.col("totaloutstanddebtvalue_39A").max().alias("totaloutstanddebtvalue_39A_max"),
])

master_data = train_static.join(
    person_flat, how="left", on="case_id"
).join(
    cb_a_1_flat, how="left", on="case_id"
)

train_static_feats = master_data.with_columns([
    (pl.col("totaldebt_9A") / (pl.col("mainoccupationinc_384A") + 1))
        .alias("debt_to_income_ratio"),
    
    (pl.col("annuity_780A") / (pl.col("mainoccupationinc_384A") + 1))
        .alias("annuity_to_income_ratio"),
    
    (pl.col("totaldebtoverduevalue_178A_max") / (pl.col("totaloutstanddebtvalue_39A_max") + 1))
        .alias("overdue_to_total_debt_ratio"),
])

In [10]:
# Debt-to-Income Ratio (DTI) - Chỉ số vàng trong tín dụng
# 1.1  lấy thu nhập của NGƯỜI VAY CHÍNH (num_group1 = 0)
person_flat = test_person_1.filter(pl.col("num_group1") == 0).select([
    "case_id", 
    "mainoccupationinc_384A"
])

# 1.2  nhóm lấy Tổng (hoặc Max) nợ quá hạn
# (Giả định cột này nằm ở bảng credit_bureau_a_1 như code bạn viết)
cb_a_1_flat = test_credit_bureau_a_1.group_by("case_id").agg([
    pl.col("totaldebtoverduevalue_178A").max().alias("totaldebtoverduevalue_178A_max"),
    pl.col("totaloutstanddebtvalue_39A").max().alias("totaloutstanddebtvalue_39A_max"),
])

master_data = test_static.join(
    person_flat, how="left", on="case_id"
).join(
    cb_a_1_flat, how="left", on="case_id"
)

test_static_feats = master_data.with_columns([
    (pl.col("totaldebt_9A") / (pl.col("mainoccupationinc_384A") + 1))
        .alias("debt_to_income_ratio"),
    
    (pl.col("annuity_780A") / (pl.col("mainoccupationinc_384A") + 1))
        .alias("annuity_to_income_ratio"),
    
    (pl.col("totaldebtoverduevalue_178A_max") / (pl.col("totaloutstanddebtvalue_39A_max") + 1))
        .alias("overdue_to_total_debt_ratio"),
])

#### Payment Behavior Deterioration (Xu hướng xấu đi của hành vi thanh toán)


- ta sẽ xem xét xu hướng hành vi nếu mà số ngày quá hạn trung bình của 3 tháng gần nhất lớn hơn nhiều lần với số ngày quá hạn trong 24 tháng vừa qua điều này chứng tỏ tình hình tài chính kinh tế ngày càng tệ đi.
- Xem xét Trung bình của các đỉnh điểm trễ hạn trong 9 tháng qua. Nếu chỉ số này cao chứng tỏ khách hàng này có thói quen chiến dụng vốn và dòng tiền đang bất ổn. Dù chưa chạm mức nợ xấu nhưng hành vì này kéo dài rồng rả 3 quý cũng là 1 rủi ro.
- Xem xét về thời gian trong khoảng thời gian 2 năm qua một người đã ao lần chuyển tiền trả nợ

In [11]:
train_credit_bureau_trend = train_static.group_by("case_id").agg([
    (pl.col("avgdbddpdlast3m_4187120P") - pl.col("avgdbddpdlast24m_3658932P"))
        .mean().alias("dpd_trend_3m_vs_24m"),

    pl.col("avgmaxdpdlast9m_3716943P").max().alias("max_dpd_9m"),

    (pl.col("cntpmts24_3658933L").cast(pl.Float32) / 24)
        .mean().alias("payment_frequency_ratio"),

    pl.col("pctinstlsallpaidlate1d_3546856L")
        .cast(pl.Float32)
        .mean().alias("late_payment_pct"),

    pl.col("pctinstlsallpaidlat10d_839L")
        .cast(pl.Float32)
        .mean().alias("very_late_payment_pct"),
])

In [12]:
test_credit_bureau_trend = test_static.group_by("case_id").agg([
    (pl.col("avgdbddpdlast3m_4187120P") - pl.col("avgdbddpdlast24m_3658932P"))
        .mean().alias("dpd_trend_3m_vs_24m"),

    pl.col("avgmaxdpdlast9m_3716943P").max().alias("max_dpd_9m"),

    (pl.col("cntpmts24_3658933L").cast(pl.Float32) / 24)
        .mean().alias("payment_frequency_ratio"),

    pl.col("pctinstlsallpaidlate1d_3546856L")
        .cast(pl.Float32)
        .mean().alias("late_payment_pct"),

    pl.col("pctinstlsallpaidlat10d_839L")
        .cast(pl.Float32)
        .mean().alias("very_late_payment_pct"),
])

#### Credit Utilization & Exposure (Mức độ sử dụng tín dụng)

In [13]:
train_credacc = train_applprev_1.group_by("case_id").agg([
    pl.col("credacc_actualbalance_314A").mean().alias("credacc_actualbalance_314A_mean"),
    pl.col("credacc_credlmt_575A").max().alias("credacc_credlmt_575A_mean"),
])


train_credit_bureau_b_1_for_exposure = train_credit_bureau_b_1.group_by("case_id").agg([
    pl.col("residualamount_3940956A").mean().alias("residualamount_3940956A_mean"),
    pl.col("credlmt_1052A").max().alias("credlmt_1052A_mean"),
])


mater_data = train_credit_bureau_a_1.join(
    train_credacc, how="left", on="case_id"
).join(
    train_credit_bureau_b_1_for_exposure, how="left", on="case_id"
)


train_credit_exposure = mater_data.group_by("case_id").agg([
    # Utilization ratio: Số dư thực tế / Hạn mức
    (pl.col("credacc_actualbalance_314A_mean") / (pl.col("credacc_credlmt_575A_mean") + 1))
        .mean().alias("credit_utilization_ratio"),
    
    # Số hợp đồng tín dụng đang active
    (pl.col("numberofcontrsvalue_258L")).max().alias("active_contracts_count"),
    
    # Tổng exposure so với thu nhập
    pl.col("totalamount_996A").max().alias("total_active_credit_exposure"),
    
    # Tỷ lệ residual còn lại (nợ gốc còn lại)
    (pl.col("residualamount_3940956A_mean") / (pl.col("credlmt_1052A_mean") + 1))
        .mean().alias("residual_ratio"),
])

In [14]:
test_credacc = test_applprev_1.group_by("case_id").agg([
    pl.col("credacc_actualbalance_314A").mean().alias("credacc_actualbalance_314A_mean"),
    pl.col("credacc_credlmt_575A").max().alias("credacc_credlmt_575A_mean"),
])


test_credit_bureau_b_1_for_exposure = test_credit_bureau_b_1.group_by("case_id").agg([
    pl.col("residualamount_3940956A").mean().alias("residualamount_3940956A_mean"),
    pl.col("credlmt_1052A").max().alias("credlmt_1052A_mean"),
])


mater_data = test_credit_bureau_a_1.join(
    test_credacc, how="left", on="case_id"
).join(
    test_credit_bureau_b_1_for_exposure, how="left", on="case_id"
)


test_credit_exposure = mater_data.group_by("case_id").agg([
    # Utilization ratio: Số dư thực tế / Hạn mức
    (pl.col("credacc_actualbalance_314A_mean") / (pl.col("credacc_credlmt_575A_mean") + 1))
        .mean().alias("credit_utilization_ratio"),
    
    # Số hợp đồng tín dụng đang active
    (pl.col("numberofcontrsvalue_258L")).max().alias("active_contracts_count"),
    
    # Tổng exposure so với thu nhập
    pl.col("totalamount_996A").max().alias("total_active_credit_exposure"),
    
    # Tỷ lệ residual còn lại (nợ gốc còn lại)
    (pl.col("residualamount_3940956A_mean") / (pl.col("credlmt_1052A_mean") + 1))
        .mean().alias("residual_ratio"),
])

#### Application Fraud & Behavioral Signals (Tín hiệu gian lận)

In [15]:
train_fraud_signals = train_static.select([
    "case_id",

    pl.col("clientscnt_304L").cast(pl.Float32, strict=False).alias("same_phone_clients"),
    pl.col("clientscnt_136L").cast(pl.Float32, strict=False).alias("same_email_clients"),
    pl.col("applicationscnt_1086L").cast(pl.Float32, strict=False).alias("same_phone_apps"),
    pl.col("applications30d_658L").cast(pl.Float32, strict=False).alias("apps_last_30d"),
    pl.col("applicationscnt_464L").cast(pl.Float32, strict=False).alias("same_employer_apps"),
]).with_columns([
    (
        (pl.col("same_phone_clients") > 2).cast(pl.Int32)
        + (pl.col("same_email_clients") > 2).cast(pl.Int32)
        + (pl.col("apps_last_30d") > 3).cast(pl.Int32)
    ).alias("fraud_signal_score")
])

In [16]:
test_fraud_signals = test_static.select([
    "case_id",

    pl.col("clientscnt_304L").cast(pl.Float32, strict=False).alias("same_phone_clients"),
    pl.col("clientscnt_136L").cast(pl.Float32, strict=False).alias("same_email_clients"),
    pl.col("applicationscnt_1086L").cast(pl.Float32, strict=False).alias("same_phone_apps"),
    pl.col("applications30d_658L").cast(pl.Float32, strict=False).alias("apps_last_30d"),
    pl.col("applicationscnt_464L").cast(pl.Float32, strict=False).alias("same_employer_apps"),
]).with_columns([
    (
        (pl.col("same_phone_clients") > 2).cast(pl.Int32)
        + (pl.col("same_email_clients") > 2).cast(pl.Int32)
        + (pl.col("apps_last_30d") > 3).cast(pl.Int32)
    ).alias("fraud_signal_score")
])

####  Employment & Income Stability (Mức độ ổn định thu nhập)

In [17]:
# Từ tax registry — thu nhập có đều không?



train_tax_stability = train_static_cb.group_by("case_id").agg([
    # Số lần có tax deduction (proxy cho tháng đi làm)
    pl.col("pmtcount_693L").max().alias("tax_payment_count"),
    
    # Biến động thu nhập (std / mean)
    (pl.col("pmtaverage_3A").std() / (pl.col("pmtaverage_3A").mean() + 1))
        .alias("income_volatility"),  # Cao = thu nhập không ổn định
    
    # Tổng thu nhập đã khai thuế
    pl.col("pmtssum_45A").sum().alias("total_declared_income"),
])

# So sánh thu nhập khai báo vs thu nhập trong đơn vay
# Nếu chênh lệch lớn = rủi ro gian lận hoặc over-leverage

In [18]:
# Từ tax registry — thu nhập có đều không?
test_tax_stability = test_static_cb.group_by("case_id").agg([
    # Số lần có tax deduction (proxy cho tháng đi làm)
    pl.col("pmtcount_693L").max().alias("tax_payment_count"),
    
    # Biến động thu nhập (std / mean)
    (pl.col("pmtaverage_3A").std() / (pl.col("pmtaverage_3A").mean() + 1))
        .alias("income_volatility"),  # Cao = thu nhập không ổn định
    
    # Tổng thu nhập đã khai thuế
    pl.col("pmtssum_45A").sum().alias("total_declared_income"),
])

# So sánh thu nhập khai báo vs thu nhập trong đơn vay
# Nếu chênh lệch lớn = rủi ro gian lận hoặc over-leverage

#### Recency-Frequency-Monetary (RFM) trên lịch sử tín dụng

In [19]:
train_credit_bureau_b_2_enhanced = train_credit_bureau_b_2.group_by("case_id").agg([
    # Frequency: số lần quá hạn
    pl.col("pmts_dpdvalue_108P")
        .filter(pl.col("pmts_dpdvalue_108P") > 0)
        .count()
        .alias("overdue_payment_count"),

    # Monetary: tổng số tiền quá hạn
    pl.col("pmts_pmtsoverdue_635A")
        .sum()
        .alias("total_overdue_amount"),

    # Severity: DPD > 31 ngày
    (pl.col("pmts_dpdvalue_108P") > 31)
        .sum()
        .alias("severe_dpd_count"),

    # DPD > 90 ngày
    (pl.col("pmts_dpdvalue_108P") > 90)
        .max()
        .cast(pl.Int8)
        .alias("has_90dpd_ever"),
])

In [20]:
test_credit_bureau_b_2_enhanced = test_credit_bureau_b_2.group_by("case_id").agg([
    # Frequency: số lần quá hạn
    pl.col("pmts_dpdvalue_108P")
        .filter(pl.col("pmts_dpdvalue_108P") > 0)
        .count()
        .alias("overdue_payment_count"),

    # Monetary: tổng số tiền quá hạn
    pl.col("pmts_pmtsoverdue_635A")
        .sum()
        .alias("total_overdue_amount"),

    # Severity: DPD > 31 ngày
    (pl.col("pmts_dpdvalue_108P") > 31)
        .sum()
        .alias("severe_dpd_count"),

    # DPD > 90 ngày
    (pl.col("pmts_dpdvalue_108P") > 90)
        .max()
        .cast(pl.Int8)
        .alias("has_90dpd_ever"),
])

####  Collateral Quality & Loan-to-Value (Chất lượng tài sản đảm bảo)

In [21]:
collateral_flat = train_credit_bureau_a_2.group_by("case_id").agg([
    pl.col("collater_valueofguarantee_1124L")
        .cast(pl.Float32)
        .max()
        .alias("max_collateral_value"),

    pl.col("collater_valueofguarantee_1124L")
        .cast(pl.Float32)
        .sum()
        .alias("total_collateral_value"),

    (pl.col("collater_valueofguarantee_1124L").cast(pl.Float32) > 0)
        .max()
        .cast(pl.Int8)
        .alias("has_collateral"),
])

train_collateral = (
    collateral_flat
    .join(
        train_static.select(["case_id", "credamount_770A"]),
        how="left",
        on="case_id"
    )
    .with_columns([
        (pl.col("credamount_770A") / (pl.col("max_collateral_value") + 1))
            .alias("loan_to_value_ratio")
    ])
    .select([
        "case_id",
        "max_collateral_value",
        "total_collateral_value",
        "has_collateral",
        "loan_to_value_ratio",
    ])
)

In [22]:
collateral_flat = test_credit_bureau_a_2.group_by("case_id").agg([
    pl.col("collater_valueofguarantee_1124L")
        .cast(pl.Float32)
        .max()
        .alias("max_collateral_value"),

    pl.col("collater_valueofguarantee_1124L")
        .cast(pl.Float32)
        .sum()
        .alias("total_collateral_value"),

    (pl.col("collater_valueofguarantee_1124L").cast(pl.Float32) > 0)
        .max()
        .cast(pl.Int8)
        .alias("has_collateral"),
])

test_collateral = (
    collateral_flat
    .join(
        train_static.select(["case_id", "credamount_770A"]),
        how="left",
        on="case_id"
    )
    .with_columns([
        (pl.col("credamount_770A") / (pl.col("max_collateral_value") + 1))
            .alias("loan_to_value_ratio")
    ])
    .select([
        "case_id",
        "max_collateral_value",
        "total_collateral_value",
        "has_collateral",
        "loan_to_value_ratio",
    ])
)

#### Temporal Risk Features (Rủi ro theo thời gian)

In [23]:
train_basetable = train_basetable.with_columns(
    pl.col("date_decision").cast(pl.Date)
).with_columns([
    # Month
    pl.col("date_decision").dt.month().alias("application_month"),

    # Quarter
    pl.col("date_decision").dt.quarter().alias("application_quarter"),

    # Week of year
    pl.col("date_decision").dt.week().alias("application_week"),

    # Day of month
    pl.col("date_decision").dt.day().alias("application_day"),

    # Weekday (Mon=1 ... Sun=7)
    pl.col("date_decision").dt.weekday().alias("application_weekday"),

    # Weekend
    (pl.col("date_decision").dt.weekday() >= 6)
        .cast(pl.Int8)
        .alias("is_weekend"),

    # End of month
    (pl.col("date_decision").dt.day() >= 25)
        .cast(pl.Int8)
        .alias("is_end_of_month"),

    # Beginning of month
    (pl.col("date_decision").dt.day() <= 5)
        .cast(pl.Int8)
        .alias("is_begin_month"),

    # Middle of month
    (
        (pl.col("date_decision").dt.day() >= 10) &
        (pl.col("date_decision").dt.day() <= 20)
    ).cast(pl.Int8).alias("is_mid_month"),
])

In [24]:
test_basetable = test_basetable.with_columns(
    pl.col("date_decision").cast(pl.Date)
).with_columns([
    # Month
    pl.col("date_decision").dt.month().alias("application_month"),

    # Quarter
    pl.col("date_decision").dt.quarter().alias("application_quarter"),

    # Week of year
    pl.col("date_decision").dt.week().alias("application_week"),

    # Day of month
    pl.col("date_decision").dt.day().alias("application_day"),

    # Weekday (Mon=1 ... Sun=7)
    pl.col("date_decision").dt.weekday().alias("application_weekday"),

    # Weekend
    (pl.col("date_decision").dt.weekday() >= 6)
        .cast(pl.Int8)
        .alias("is_weekend"),

    # End of month
    (pl.col("date_decision").dt.day() >= 25)
        .cast(pl.Int8)
        .alias("is_end_of_month"),

    # Beginning of month
    (pl.col("date_decision").dt.day() <= 5)
        .cast(pl.Int8)
        .alias("is_begin_month"),

    # Middle of month
    (
        (pl.col("date_decision").dt.day() >= 10) &
        (pl.col("date_decision").dt.day() <= 20)
    ).cast(pl.Int8).alias("is_mid_month"),
])

#### Tổng hợp bảng các thông tin:
- train_static_feats
- train_credit_bureau_trend
- train_credit_exposure
- train_fraud_signals
- train_tax_stability
- train_credit_bureau_b_2_enhanced
- train_collateral 

In [25]:
# Dùng collect_schema().names() thay vì .columns trên LazyFrame
selected_static_cols = [
    col for col in train_static.collect_schema().names()
    if col[-1] in ("A", "M")
]
print(selected_static_cols)

selected_static_cb_cols = [
    col for col in train_static_cb.collect_schema().names()
    if col[-1] in ("A", "M")
]
print(selected_static_cb_cols)

# Join all tables together
data = train_basetable.join(
    train_static.select(["case_id"] + selected_static_cols), how="left", on="case_id"
).join(
    train_static_cb.select(["case_id"] + selected_static_cb_cols), how="left", on="case_id"
).join(
    train_person_1_feats_1, how="left", on="case_id"
).join(
    train_person_1_feats_2, how="left", on="case_id"
).join(
    train_credit_bureau_b_2_feats, how="left", on="case_id"
).join(
    train_static_feats, how="left", on="case_id"
).join(
    train_credit_bureau_trend, how="left", on="case_id"
).join(
    train_credit_exposure, how="left", on="case_id"
).join(
    train_fraud_signals, how="left", on="case_id"
).join(
    train_tax_stability, how="left", on="case_id"
).join(
    train_credit_bureau_b_2_enhanced, how="left", on="case_id"
).join(
    train_collateral, how="left", on="case_id"
)

['amtinstpaidbefduel24m_4187115A', 'annuity_780A', 'annuitynextmonth_57A', 'avginstallast24m_3658937A', 'avglnamtstart24m_4525187A', 'avgoutstandbalancel6m_4187114A', 'avgpmtlast12m_4525200A', 'credamount_770A', 'currdebt_22A', 'currdebtcredtyperange_828A', 'disbursedcredamount_1113A', 'downpmt_116A', 'inittransactionamount_650A', 'lastapprcommoditycat_1041M', 'lastapprcommoditytypec_5251766M', 'lastapprcredamount_781A', 'lastcancelreason_561M', 'lastotherinc_902A', 'lastotherlnsexpense_631A', 'lastrejectcommoditycat_161M', 'lastrejectcommodtypec_5251769M', 'lastrejectcredamount_222A', 'lastrejectreason_759M', 'lastrejectreasonclient_4145040M', 'maininc_215A', 'maxannuity_159A', 'maxannuity_4075009A', 'maxdebt4_972A', 'maxinstallast24m_3658928A', 'maxlnamtstart6m_4525199A', 'maxoutstandbalancel12m_4187113A', 'maxpmtlast3m_4525190A', 'previouscontdistrict_112M', 'price_1097A', 'sumoutstandtotal_3546847A', 'sumoutstandtotalest_4493215A', 'totaldebt_9A', 'totalsettled_863A', 'totinstallas

In [26]:

data_submission = test_basetable.join(
    test_static.select(["case_id"]+selected_static_cols), how="left", on="case_id"
).join(
    test_static_cb.select(["case_id"]+selected_static_cb_cols), how="left", on="case_id"
).join(
    test_person_1_feats_1, how="left", on="case_id"
).join(
    test_person_1_feats_2, how="left", on="case_id"
).join(
    test_credit_bureau_b_2_feats, how="left", on="case_id"
).join(
    test_static_feats, how="left", on="case_id"
).join(
    test_credit_bureau_trend, how="left", on="case_id"
).join(
    test_credit_exposure, how="left", on="case_id"
).join(
    test_fraud_signals, how="left", on="case_id"
).join(
    test_tax_stability, how="left", on="case_id"
).join(
    test_credit_bureau_b_2_enhanced, how="left", on="case_id"
).join(
    test_collateral, how="left", on="case_id"
)

In [27]:
# Collect nếu data còn là LazyFrame
if isinstance(data, pl.LazyFrame):
    data = data.collect()

# Split case_id
case_ids = data["case_id"].unique().shuffle(seed=1).to_list()

case_ids_train, case_ids_test = train_test_split(
    case_ids,
    train_size=0.6,
    random_state=1
)

case_ids_valid, case_ids_test = train_test_split(
    case_ids_test,
    train_size=0.5,
    random_state=1
)

# Lấy tất cả feature, chỉ bỏ các cột metadata / label
excluded_cols = {
    "case_id",
    "target",
    "WEEK_NUM",
    "date_decision",
}

cols_pred = [
    col for col in data.columns
    if col not in excluded_cols
]

print("Number of features:", len(cols_pred))
print(cols_pred)

def from_polars_to_pandas(case_ids):
    df = data.filter(
        pl.col("case_id").is_in(case_ids)
    )

    base = df[["case_id", "WEEK_NUM", "target"]].to_pandas()
    X = df[cols_pred].to_pandas()
    y = df["target"].to_pandas()

    return base, X, y

base_train, X_train, y_train = from_polars_to_pandas(case_ids_train)
base_valid, X_valid, y_valid = from_polars_to_pandas(case_ids_valid)
base_test, X_test, y_test = from_polars_to_pandas(case_ids_test)

# Convert object/string sang category
X_train = convert_strings(X_train)
X_valid = convert_strings(X_valid)
X_test = convert_strings(X_test)

print(f"Train: {X_train.shape}")
print(f"Valid: {X_valid.shape}")
print(f"Test : {X_test.shape}")

Number of features: 262
['MONTH', 'application_month', 'application_quarter', 'application_week', 'application_day', 'application_weekday', 'is_weekend', 'is_end_of_month', 'is_begin_month', 'is_mid_month', 'amtinstpaidbefduel24m_4187115A', 'annuity_780A', 'annuitynextmonth_57A', 'avginstallast24m_3658937A', 'avglnamtstart24m_4525187A', 'avgoutstandbalancel6m_4187114A', 'avgpmtlast12m_4525200A', 'credamount_770A', 'currdebt_22A', 'currdebtcredtyperange_828A', 'disbursedcredamount_1113A', 'downpmt_116A', 'inittransactionamount_650A', 'lastapprcommoditycat_1041M', 'lastapprcommoditytypec_5251766M', 'lastapprcredamount_781A', 'lastcancelreason_561M', 'lastotherinc_902A', 'lastotherlnsexpense_631A', 'lastrejectcommoditycat_161M', 'lastrejectcommodtypec_5251769M', 'lastrejectcredamount_222A', 'lastrejectreason_759M', 'lastrejectreasonclient_4145040M', 'maininc_215A', 'maxannuity_159A', 'maxannuity_4075009A', 'maxdebt4_972A', 'maxinstallast24m_3658928A', 'maxlnamtstart6m_4525199A', 'maxoutst

## Training Xgboost 

Trong phần này chỉ sử dụng xgboost để thử nghiệm train

In [28]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score


def prepare_xgb_data(X_train, X_valid, X_test, X_submission=None):
    X_train = X_train.copy()
    X_valid = X_valid.copy()
    X_test = X_test.copy()

    datasets = [X_train, X_valid, X_test]
    if X_submission is not None:
        X_submission = X_submission.copy()
        datasets.append(X_submission)

    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in cat_cols:
        all_values = pd.concat(
            [df[col].astype("object") for df in datasets],
            axis=0
        ).fillna("Unknown")

        categories = all_values.astype("category").cat.categories

        dtype = pd.CategoricalDtype(categories=categories)

        for df in datasets:
            df[col] = df[col].astype("object").fillna("Unknown").astype(dtype).cat.codes

    if X_submission is not None:
        return X_train, X_valid, X_test, X_submission

    return X_train, X_valid, X_test

In [29]:
X_train_xgb, X_valid_xgb, X_test_xgb = prepare_xgb_data(
    X_train, X_valid, X_test
)

model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="auc",
    device="cuda",
    tree_method="hist",
    random_state=1,
    n_jobs=-1,
    early_stopping_rounds=50,
)

model.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_valid_xgb, y_valid)],
    verbose=50,
)

valid_pred = model.predict_proba(X_valid_xgb)[:, 1]
print("Valid AUC:", roc_auc_score(y_valid, valid_pred))

[0]	validation_0-auc:0.66068
[50]	validation_0-auc:0.76032
[100]	validation_0-auc:0.77954
[150]	validation_0-auc:0.78942
[200]	validation_0-auc:0.79469
[250]	validation_0-auc:0.79826
[300]	validation_0-auc:0.80104
[350]	validation_0-auc:0.80321
[400]	validation_0-auc:0.80487
[450]	validation_0-auc:0.80625
[500]	validation_0-auc:0.80744
[550]	validation_0-auc:0.80824
[600]	validation_0-auc:0.80919
[650]	validation_0-auc:0.80990
[700]	validation_0-auc:0.81074
[750]	validation_0-auc:0.81138
[800]	validation_0-auc:0.81201
[850]	validation_0-auc:0.81248
[900]	validation_0-auc:0.81280
[950]	validation_0-auc:0.81328
[999]	validation_0-auc:0.81352


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [11:00:45] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Valid AUC: 0.813535656823217


Evaluation with AUC and then comparison with the stability metric is shown below.

In [30]:
datasets = {
    "Train": (base_train, X_train_xgb),
    "Valid": (base_valid, X_valid_xgb),
    "Test": (base_test, X_test_xgb),
}

for name, (base, X) in datasets.items():
    base["score"] = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(base["target"], base["score"])
    print(f"{name:5s} AUC: {auc:.6f}")

Train AUC: 0.828579
Valid AUC: 0.813536
Test  AUC: 0.812371


In [31]:
def gini_stability(base, w_fallingrate=88.0, w_resstd=-0.5):
    gini_in_time = base.loc[:, ["WEEK_NUM", "target", "score"]]\
        .sort_values("WEEK_NUM")\
        .groupby("WEEK_NUM")[["target", "score"]]\
        .apply(lambda x: 2*roc_auc_score(x["target"], x["score"])-1).tolist()
    
    x = np.arange(len(gini_in_time))
    y = gini_in_time
    a, b = np.polyfit(x, y, 1)
    y_hat = a*x + b
    residuals = y - y_hat
    res_std = np.std(residuals)
    avg_gini = np.mean(gini_in_time)
    return avg_gini + w_fallingrate * min(0, a) + w_resstd * res_std

stability_score_train = gini_stability(base_train)
stability_score_valid = gini_stability(base_valid)
stability_score_test = gini_stability(base_test)

print(f'The stability score on the train set is: {stability_score_train}') 
print(f'The stability score on the valid set is: {stability_score_valid}') 
print(f'The stability score on the test set is: {stability_score_test}')  

The stability score on the train set is: 0.6268446782681151
The stability score on the valid set is: 0.5876456486677961
The stability score on the test set is: 0.590816601548053


## Áp dụng cách cải tiển của nhóm phát triển

1. Train LightGBM

In [32]:
# Các cột category của train
cat_cols_lgb = X_train.select_dtypes(include=["category"]).columns.tolist()

print(f"Remove {len(cat_cols_lgb)} categorical columns:")
print(cat_cols_lgb)

# Drop đồng bộ
X_train_lgb = X_train.drop(columns=cat_cols_lgb)
X_valid_lgb = X_valid.drop(columns=cat_cols_lgb)
X_test_lgb = X_test.drop(columns=cat_cols_lgb)

Remove 53 categorical columns:
['lastapprcommoditycat_1041M', 'lastapprcommoditytypec_5251766M', 'lastcancelreason_561M', 'lastrejectcommoditycat_161M', 'lastrejectcommodtypec_5251769M', 'lastrejectreason_759M', 'lastrejectreasonclient_4145040M', 'previouscontdistrict_112M', 'description_5085714M', 'education_1103M', 'education_88M', 'maritalst_385M', 'maritalst_893M', 'person_housetype', 'pmts_dpdvalue_108P_over31', 'bankacctype_710L', 'cardtype_51L', 'credtype_322L', 'datefirstoffer_1144D', 'datelastinstal40dpd_247D', 'datelastunpaid_3546854D', 'disbursementtype_67L', 'dtlastpmtallstes_4499206D', 'equalitydataagreement_891L', 'equalityempfrom_62L', 'firstclxcampaign_1125D', 'firstdatedue_489D', 'inittransactioncode_186L', 'isbidproductrequest_292L', 'isdebitcard_729L', 'lastactivateddate_801D', 'lastapplicationdate_877D', 'lastapprcommoditycat_1041M_right', 'lastapprcommoditytypec_5251766M_right', 'lastapprdate_640D', 'lastcancelreason_561M_right', 'lastdelinqdate_224D', 'lastrejectc

In [33]:
# Drop categorical columns cho LightGBM
cat_cols_lgb = X_train.select_dtypes(include=["category"]).columns.tolist()

print(f"Remove {len(cat_cols_lgb)} categorical columns:")
print(cat_cols_lgb)

X_train_lgb = X_train.drop(columns=cat_cols_lgb)
X_valid_lgb = X_valid.drop(columns=cat_cols_lgb)
X_test_lgb = X_test.drop(columns=cat_cols_lgb)

Remove 53 categorical columns:
['lastapprcommoditycat_1041M', 'lastapprcommoditytypec_5251766M', 'lastcancelreason_561M', 'lastrejectcommoditycat_161M', 'lastrejectcommodtypec_5251769M', 'lastrejectreason_759M', 'lastrejectreasonclient_4145040M', 'previouscontdistrict_112M', 'description_5085714M', 'education_1103M', 'education_88M', 'maritalst_385M', 'maritalst_893M', 'person_housetype', 'pmts_dpdvalue_108P_over31', 'bankacctype_710L', 'cardtype_51L', 'credtype_322L', 'datefirstoffer_1144D', 'datelastinstal40dpd_247D', 'datelastunpaid_3546854D', 'disbursementtype_67L', 'dtlastpmtallstes_4499206D', 'equalitydataagreement_891L', 'equalityempfrom_62L', 'firstclxcampaign_1125D', 'firstdatedue_489D', 'inittransactioncode_186L', 'isbidproductrequest_292L', 'isdebitcard_729L', 'lastactivateddate_801D', 'lastapplicationdate_877D', 'lastapprcommoditycat_1041M_right', 'lastapprcommoditytypec_5251766M_right', 'lastapprdate_640D', 'lastcancelreason_561M_right', 'lastdelinqdate_224D', 'lastrejectc

In [34]:
lgb_train = lgb.Dataset(
    X_train_lgb,
    label=y_train,
    free_raw_data=False
)

lgb_valid = lgb.Dataset(
    X_valid_lgb,
    label=y_valid,
    reference=lgb_train,
    free_raw_data=False
)

lgb_params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 5,
    "min_data_in_leaf": 50,
    "lambda_l1": 0.1,
    "lambda_l2": 5.0,
    "verbose": -1,
    "seed": 1,

    # nếu muốn thử GPU
    "device_type": "gpu",
}

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=5000,
    valid_sets=[lgb_valid],
    valid_names=["valid"],
    callbacks=[
        lgb.log_evaluation(100),
        lgb.early_stopping(200)
    ]
)

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 200 rounds
[100]	valid's auc: 0.796591
[200]	valid's auc: 0.806893
[300]	valid's auc: 0.810444
[400]	valid's auc: 0.811857
[500]	valid's auc: 0.81251
[600]	valid's auc: 0.813071
[700]	valid's auc: 0.813354
[800]	valid's auc: 0.813748
[900]	valid's auc: 0.813955
[1000]	valid's auc: 0.814163
[1100]	valid's auc: 0.814221
[1200]	valid's auc: 0.814226
[1300]	valid's auc: 0.814172
Early stopping, best iteration is:
[1156]	valid's auc: 0.814286


2. Train XGBoost

In [35]:
import xgboost as xgb

X_train_xgb, X_valid_xgb, X_test_xgb = prepare_xgb_data(
    X_train,
    X_valid,
    X_test
)

xgb_model = xgb.XGBClassifier(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=10,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=5.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    device="cuda",
    random_state=1,
    early_stopping_rounds=200,
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[(X_valid_xgb, y_valid)],
    verbose=100
)

[0]	validation_0-auc:0.69989
[100]	validation_0-auc:0.78627
[200]	validation_0-auc:0.80079
[300]	validation_0-auc:0.80623
[400]	validation_0-auc:0.80955
[500]	validation_0-auc:0.81174
[600]	validation_0-auc:0.81342
[700]	validation_0-auc:0.81465
[800]	validation_0-auc:0.81574
[900]	validation_0-auc:0.81666
[1000]	validation_0-auc:0.81708
[1100]	validation_0-auc:0.81756
[1200]	validation_0-auc:0.81804
[1300]	validation_0-auc:0.81847
[1400]	validation_0-auc:0.81876
[1500]	validation_0-auc:0.81908
[1600]	validation_0-auc:0.81932
[1700]	validation_0-auc:0.81946
[1800]	validation_0-auc:0.81959
[1900]	validation_0-auc:0.81978
[2000]	validation_0-auc:0.81990
[2100]	validation_0-auc:0.82002
[2200]	validation_0-auc:0.82006
[2300]	validation_0-auc:0.82015
[2400]	validation_0-auc:0.82028
[2500]	validation_0-auc:0.82031
[2600]	validation_0-auc:0.82028
[2638]	validation_0-auc:0.82032


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device='cuda', early_stopping_rounds=200,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=10, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=5000,
              n_jobs=None, num_parallel_tree=None, ...)

3. Train CatBoost

In [36]:
from catboost import CatBoostClassifier

cat_cols_cat = X_train.select_dtypes(include=["category", "object"]).columns.tolist()
cat_features = [X_train.columns.get_loc(c) for c in cat_cols_cat]

X_train_cat = X_train.copy()
X_valid_cat = X_valid.copy()
X_test_cat = X_test.copy()

for df in [X_train_cat, X_valid_cat, X_test_cat]:
    for col in cat_cols_cat:
        df[col] = df[col].astype("object").fillna("Unknown").astype(str)

cat_model = CatBoostClassifier(
    iterations=6000,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=1,
    task_type="GPU",
    devices="0",
    verbose=100,
    early_stopping_rounds=200,
)

cat_model.fit(
    X_train_cat,
    y_train,
    eval_set=(X_valid_cat, y_valid),
    cat_features=cat_features,
    use_best_model=True
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.5783366	best: 0.5783366 (0)	total: 605ms	remaining: 1h 27s
100:	test: 0.7678060	best: 0.7678060 (100)	total: 45.8s	remaining: 44m 33s
200:	test: 0.7846644	best: 0.7846644 (200)	total: 1m 31s	remaining: 44m 11s
300:	test: 0.7911690	best: 0.7911690 (300)	total: 2m 18s	remaining: 43m 34s
400:	test: 0.7952702	best: 0.7952702 (400)	total: 3m 3s	remaining: 42m 39s
500:	test: 0.7978735	best: 0.7978735 (500)	total: 3m 48s	remaining: 41m 47s
600:	test: 0.7995934	best: 0.7995934 (600)	total: 4m 32s	remaining: 40m 51s
700:	test: 0.8009797	best: 0.8009797 (700)	total: 5m 17s	remaining: 40m 2s
800:	test: 0.8019966	best: 0.8019966 (800)	total: 6m 1s	remaining: 39m 8s
900:	test: 0.8030088	best: 0.8030088 (900)	total: 6m 46s	remaining: 38m 22s
1000:	test: 0.8039163	best: 0.8039163 (1000)	total: 7m 30s	remaining: 37m 30s
1100:	test: 0.8047122	best: 0.8047122 (1100)	total: 8m 15s	remaining: 36m 42s
1200:	test: 0.8053139	best: 0.8053139 (1200)	total: 8m 59s	remaining: 35m 56s
1300:	test: 0.805

CatBoostClassifier(depth=6, devices='0', early_stopping_rounds=200, eval_metric='AUC', iterations=6000, learning_rate=0.03, loss_function='Logloss', random_seed=1, task_type='GPU', verbose=100)

4. Dự đoán validation/test cho từng model

In [37]:
pred_lgb_train = lgb_model.predict(
    X_train_lgb,
    num_iteration=lgb_model.best_iteration
)

pred_lgb_valid = lgb_model.predict(
    X_valid_lgb,
    num_iteration=lgb_model.best_iteration
)

pred_lgb_test = lgb_model.predict(
    X_test_lgb,
    num_iteration=lgb_model.best_iteration
)

# =====================
# XGBoost predictions
# =====================
pred_xgb_train = xgb_model.predict_proba(X_train_xgb)[:, 1]
pred_xgb_valid = xgb_model.predict_proba(X_valid_xgb)[:, 1]
pred_xgb_test = xgb_model.predict_proba(X_test_xgb)[:, 1]

# =====================
# CatBoost predictions
# =====================
pred_cat_train = cat_model.predict_proba(X_train_cat)[:, 1]
pred_cat_valid = cat_model.predict_proba(X_valid_cat)[:, 1]
pred_cat_test = cat_model.predict_proba(X_test_cat)[:, 1]

# =====================
# AUC từng model
# =====================
print("=" * 60)
print("AUC SCORE")
print("=" * 60)

print(f"LGB Train AUC : {roc_auc_score(y_train, pred_lgb_train):.6f}")
print(f"LGB Valid AUC : {roc_auc_score(y_valid, pred_lgb_valid):.6f}")
print(f"LGB Test  AUC : {roc_auc_score(y_test, pred_lgb_test):.6f}")
print("-" * 60)

print(f"XGB Train AUC : {roc_auc_score(y_train, pred_xgb_train):.6f}")
print(f"XGB Valid AUC : {roc_auc_score(y_valid, pred_xgb_valid):.6f}")
print(f"XGB Test  AUC : {roc_auc_score(y_test, pred_xgb_test):.6f}")
print("-" * 60)

print(f"CAT Train AUC : {roc_auc_score(y_train, pred_cat_train):.6f}")
print(f"CAT Valid AUC : {roc_auc_score(y_valid, pred_cat_valid):.6f}")
print(f"CAT Test  AUC : {roc_auc_score(y_test, pred_cat_test):.6f}")
print("=" * 60)

# =====================
# Stability từng model
# =====================
def evaluate_stability(base, pred):
    tmp = base.copy()
    tmp["score"] = pred
    return gini_stability(tmp)

print("GINI STABILITY")
print("=" * 60)

print(f"LGB Train Stability : {evaluate_stability(base_train, pred_lgb_train):.6f}")
print(f"LGB Valid Stability : {evaluate_stability(base_valid, pred_lgb_valid):.6f}")
print(f"LGB Test  Stability : {evaluate_stability(base_test, pred_lgb_test):.6f}")
print("-" * 60)

print(f"XGB Train Stability : {evaluate_stability(base_train, pred_xgb_train):.6f}")
print(f"XGB Valid Stability : {evaluate_stability(base_valid, pred_xgb_valid):.6f}")
print(f"XGB Test  Stability : {evaluate_stability(base_test, pred_xgb_test):.6f}")
print("-" * 60)

print(f"CAT Train Stability : {evaluate_stability(base_train, pred_cat_train):.6f}")
print(f"CAT Valid Stability : {evaluate_stability(base_valid, pred_cat_valid):.6f}")
print(f"CAT Test  Stability : {evaluate_stability(base_test, pred_cat_test):.6f}")
print("=" * 60)

AUC SCORE
LGB Train AUC : 0.896496
LGB Valid AUC : 0.814286
LGB Test  AUC : 0.813540
------------------------------------------------------------
XGB Train AUC : 0.876885
XGB Valid AUC : 0.820347
XGB Test  AUC : 0.820179
------------------------------------------------------------
CAT Train AUC : 0.908033
CAT Valid AUC : 0.815632
CAT Test  AUC : 0.817405
GINI STABILITY
LGB Train Stability : 0.780657
LGB Valid Stability : 0.585093
LGB Test  Stability : 0.591068
------------------------------------------------------------
XGB Train Stability : 0.736531
XGB Valid Stability : 0.596294
XGB Test  Stability : 0.606620
------------------------------------------------------------
CAT Train Stability : 0.804999
CAT Valid Stability : 0.586831
CAT Test  Stability : 0.598554


5. Weighted Ensemble

In [38]:
weights = {
    "lgb": 0.40,
    "xgb": 0.35,
    "cat": 0.25,
}

pred_valid_ens = (
    weights["lgb"] * pred_lgb_valid +
    weights["xgb"] * pred_xgb_valid +
    weights["cat"] * pred_cat_valid
)

print("Ensemble Valid AUC:", roc_auc_score(y_valid, pred_valid_ens))

Ensemble Valid AUC: 0.821544938740528


6. Tìm trọng số tốt nhất đơn giản

In [39]:
best_stability = -np.inf
best_auc = -np.inf
best_weights = None

for w_lgb in np.arange(0.0, 1.01, 0.05):
    for w_xgb in np.arange(0.0, 1.01 - w_lgb, 0.05):
        w_cat = round(1.0 - w_lgb - w_xgb, 2)

        pred = (
            w_lgb * pred_lgb_valid +
            w_xgb * pred_xgb_valid +
            w_cat * pred_cat_valid
        )

        auc = roc_auc_score(y_valid, pred)

        tmp = base_valid.copy()
        tmp["score"] = pred

        stability = gini_stability(tmp)

        # Ưu tiên Stability, nếu bằng nhau thì chọn AUC lớn hơn
        if (
            stability > best_stability or
            (np.isclose(stability, best_stability) and auc > best_auc)
        ):
            best_stability = stability
            best_auc = auc
            best_weights = (w_lgb, w_xgb, w_cat)

print("=" * 60)
print(f"Best Stability : {best_stability:.6f}")
print(f"Best AUC       : {best_auc:.6f}")
print(
    f"Best Weights   : "
    f"LGB={best_weights[0]:.2f}, "
    f"XGB={best_weights[1]:.2f}, "
    f"CAT={best_weights[2]:.2f}"
)
print("=" * 60)

Best Stability : 0.600947
Best AUC       : 0.822314
Best Weights   : LGB=0.05, XGB=0.60, CAT=0.35


7. Đánh giá train/valid/test bằng ensemble

In [40]:
w_lgb, w_xgb, w_cat = best_weights

pred_lgb_train = lgb_model.predict(
    X_train_lgb,
    num_iteration=lgb_model.best_iteration
)
pred_lgb_valid = lgb_model.predict(
    X_valid_lgb,
    num_iteration=lgb_model.best_iteration
)
pred_lgb_test = lgb_model.predict(
    X_test_lgb,
    num_iteration=lgb_model.best_iteration
)

pred_xgb_train = xgb_model.predict_proba(X_train_xgb)[:, 1]
pred_xgb_valid = xgb_model.predict_proba(X_valid_xgb)[:, 1]
pred_xgb_test = xgb_model.predict_proba(X_test_xgb)[:, 1]

pred_cat_train = cat_model.predict_proba(X_train_cat)[:, 1]
pred_cat_valid = cat_model.predict_proba(X_valid_cat)[:, 1]
pred_cat_test = cat_model.predict_proba(X_test_cat)[:, 1]

base_train = base_train.copy()
base_valid = base_valid.copy()
base_test = base_test.copy()

base_train["score"] = (
    w_lgb * pred_lgb_train +
    w_xgb * pred_xgb_train +
    w_cat * pred_cat_train
)

base_valid["score"] = (
    w_lgb * pred_lgb_valid +
    w_xgb * pred_xgb_valid +
    w_cat * pred_cat_valid
)

base_test["score"] = (
    w_lgb * pred_lgb_test +
    w_xgb * pred_xgb_test +
    w_cat * pred_cat_test
)

print(f"Train AUC: {roc_auc_score(base_train['target'], base_train['score']):.6f}")
print(f"Valid AUC: {roc_auc_score(base_valid['target'], base_valid['score']):.6f}")
print(f"Test  AUC: {roc_auc_score(base_test['target'], base_test['score']):.6f}")

print(f"Train Stability: {gini_stability(base_train):.6f}")
print(f"Valid Stability: {gini_stability(base_valid):.6f}")
print(f"Test  Stability: {gini_stability(base_test):.6f}")

Train AUC: 0.904218
Valid AUC: 0.822314
Test  AUC: 0.823108
Train Stability: 0.797335
Valid Stability: 0.600947
Test  Stability: 0.613038


## Submission

Scoring the submission dataset is below, we need to take care of new categories. Then we save the score as a last step. 

In [41]:
if isinstance(data_submission, pl.LazyFrame):
    data_submission = data_submission.collect()

# Bảo đảm submission có đủ cols_pred
missing_cols = [c for c in cols_pred if c not in data_submission.columns]
for c in missing_cols:
    data_submission = data_submission.with_columns(pl.lit(None).alias(c))

# Base submission dataframe
X_submission = data_submission[cols_pred].to_pandas()
X_submission = convert_strings(X_submission)
X_submission_lgb = X_submission.drop(columns=cat_cols_lgb)

# =====================
# LightGBM
# =====================
lgb_features = X_train_lgb.columns.tolist()

X_submission_lgb = X_submission.reindex(columns=lgb_features)

for col in X_submission_lgb.columns:
    X_submission_lgb[col] = pd.to_numeric(
        X_submission_lgb[col],
        errors="coerce"
    )

X_submission_lgb = X_submission_lgb.fillna(-1).astype("float32")

print(
    "LGB bad dtypes:",
    X_submission_lgb.select_dtypes(include=["object", "category"]).columns.tolist()
)

# =====================
# XGBoost
# =====================
_, _, _, X_submission_xgb = prepare_xgb_data(
    X_train,
    X_valid,
    X_test,
    X_submission,
)

# Bảo đảm XGBoost không còn object/category
bad_xgb_cols = X_submission_xgb.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in bad_xgb_cols:
    X_submission_xgb[col] = (
        X_submission_xgb[col]
        .astype("category")
        .cat.codes
        .astype("int32")
    )

X_submission_xgb = X_submission_xgb.reindex(columns=X_train_xgb.columns)

print(
    "XGB bad dtypes:",
    X_submission_xgb.select_dtypes(include=["object", "category"]).columns.tolist()
)

# =====================
# CatBoost
# =====================
cat_features_all = X_train_cat.columns.tolist()

X_submission_cat = X_submission.reindex(columns=cat_features_all)

for col in cat_cols_cat:
    if col in X_submission_cat.columns:
        X_submission_cat[col] = (
            X_submission_cat[col]
            .astype("object")
            .fillna("Unknown")
            .astype(str)
        )

# Các cột không phải categorical thì ép numeric
for col in X_submission_cat.columns:
    if col not in cat_cols_cat:
        X_submission_cat[col] = pd.to_numeric(
            X_submission_cat[col],
            errors="coerce"
        ).fillna(-1)

# =====================
# Predict
# =====================
pred_lgb_sub = lgb_model.predict(
    X_submission_lgb,
    num_iteration=lgb_model.best_iteration
)

pred_xgb_sub = xgb_model.predict_proba(X_submission_xgb)[:, 1]

pred_cat_sub = cat_model.predict_proba(X_submission_cat)[:, 1]

y_submission_pred = (
    w_lgb * pred_lgb_sub +
    w_xgb * pred_xgb_sub +
    w_cat * pred_cat_sub
)

print("Submission prediction shape:", y_submission_pred.shape)

LGB bad dtypes: []


/tmp/ipykernel_24/66576272.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].astype("object").fillna("Unknown").astype(dtype).cat.codes


XGB bad dtypes: []
Submission prediction shape: (10,)


/tmp/ipykernel_24/3424917591.py:76: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("Unknown")


In [42]:
submission = data_submission[["case_id"]].to_pandas()
submission["score"] = y_submission_pred
submission.to_csv("submission.csv", index=False)
submission.head()

,case_id,score
0,57543,0.017672
1,57549,0.038767
2,57551,0.010202
3,57552,0.008373
4,57569,0.099239
